## Finetuning LLMs

- `Finetuning`: Adapting a pretrained model to a `specific` task by training model on additional dataset.

- Finetuning involves the additional training of a pre-existing model, which has previously acquired patterns and features from an extensive dataset, using a smaller, domain specific dataset.

- Two types of finetuning:
  
  * `Instruction finetuning`: Training a language on a set of tasks using specific instructions. Can handle larger datasets but greater computational power is needed.

  * `Classification finetuning`: Model is trained to recognise a specific set of class labels, such as spam or not spam. Can handle narrow set of prompts.

- `Parameter Efficient Fine-Tuning(PEFT)`: More efficient than full fine-tuning. Updates only a set of parameters, effectively `freezing` the rest. This reduces number of trainable parameters, making memory requirements managable.

### Spam Classification

#### Downloading the Dataset

In [ ]:
import urllib.request
import ssl
import zipfile
import os
from pathlib import Path

url="https://archive.ics.uci.edu/static/public/228/sms+spam+collection.zip"
zip_path="sms_spam_collection.zip"
extracted_path="sms_spam_collection"
data_file_path=Path(extracted_path)/"SMSSpamCollection.tsv"

def download_and_extract(url, zip_path, extracted_path, data_file_path):
    if data_file_path.exists():
        print(f"Data file already exists at {data_file_path}. Skipping download.")
        return
    
    ssl_context=ssl._create_unverified_context()

    with urllib.request.urlopen(url, context=ssl_context) as response:
        with open(zip_path, 'wb') as out_file:
            out_file.write(response.read())

    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        zip_ref.extractall(extracted_path)

    original_file = Path(extracted_path)/"SMSSpamCollection"
    os.rename(original_file, data_file_path)
    print(f"Data file downloaded and extracted to {data_file_path}.")

download_and_extract(url, zip_path, extracted_path, data_file_path)

#### Pre Processing the Dataset

In [ ]:
import pandas as pd
df = pd.read_csv(data_file_path, sep='\t', header=None, names=['Label', 'Text'])
df

In [ ]:
print(df["Label"].value_counts())

In [ ]:
def create_balanced_dataset(df):
    num_spam=df[df["Label"]=='spam'].shape[0]

    ham_subset=df[df["Label"]=='ham'].sample(num_spam, random_state=123)

    balanced_df=pd.concat([ham_subset, df[df["Label"]=='spam']])

    return balanced_df

balanced_df=create_balanced_dataset(df)
print(balanced_df["Label"].value_counts())

In [ ]:
balanced_df["Label"]=balanced_df["Label"].map({"ham": 0, "spam": 1})

In [ ]:
def random_split(df, train_frac, val_frac):
    # Shuffle the rows and reset the row indices
    df=df.sample(frac=1, random_state=123).reset_index(drop=True)

    train_end=int(len(df)*train_frac)
    val_end=train_end+int(len(df)*val_frac)

    train_df=df[:train_end]
    val_df=df[train_end:val_end]
    test_df=df[val_end:]

    return train_df, val_df, test_df

train_df, val_df, test_df=random_split(balanced_df, 0.7, 0.1)

In [ ]:
len(train_df)

In [ ]:
len(val_df)

In [ ]:
len(test_df)

In [ ]:
train_df.to_csv("train.csv", index=None)
val_df.to_csv("validation.csv", index=None)
test_df.to_csv("test.csv", index=None)

#### Creating Dataloaders

- `Dataset`: Stores the samples and their corresponding labels.

- `DataLoader`: Wraps an iterable around the Dataset to easy access to the samples.

- The text messages are of varying lengths, so we can pad all the messages to the length of longest message. We will pad all the messages with `endoftext` token.

- Input text -> Tokenize -> Ensure all the sentences are of same length.

In [ ]:
import torch
from torch.utils.data import Dataset

class SpamDataset(Dataset):
    def __init__(self, csv_file, tokenizer, max_length=None, pad_token_id=50256):
        self.data=pd.read_csv(csv_file)

        self.encoded_texts=[
            tokenizer.encode(text) for text in self.data["Text"]
        ]

        if max_length is None:
            self.max_length=self._longest_encoded_length()
        else:
            self.max_length=max_length

            # Truncate sequences to max_length
            self.encoded_texts=[
                encoded_text[:self.max_length] for encoded_text in self.encoded_texts
            ]

        # Pad sequences to max_length
        self.encoded_texts=[
            encoded_text+[pad_token_id]*(self.max_length-len(encoded_text))
            for encoded_text in self.encoded_texts
        ]

    def __getitem__(self, index):
        encoded=self.encoded_texts[index]
        label=self.data.iloc[index]["Label"]

        return (
            torch.tensor(encoded, dtype=torch.long),
            torch.tensor(label, dtype=torch.long)
        )

    def __len__(self):
        return len(self.data)

    def _longest_encoded_length(self):
        max_length=0
        for encoded_text in self.encoded_texts:
            text_length=len(encoded_text)
            if text_length>max_length:
                max_length=text_length
        return max_length

In [ ]:
import tiktoken

tokenizer=tiktoken.get_encoding("gpt2")

train_dataset=SpamDataset(
    csv_file="train.csv",
    tokenizer=tokenizer,
    max_length=None
)

print(train_dataset.max_length)

In [ ]:
val_dataset=SpamDataset(
    csv_file="validation.csv",
    tokenizer=tokenizer,
    max_length=train_dataset.max_length
)

test_dataset=SpamDataset(
    csv_file="test.csv",
    tokenizer=tokenizer,
    max_length=train_dataset.max_length
)

In [ ]:
from torch.utils.data import DataLoader

num_workers=0
batch_size=8

train_dataloader=DataLoader(
    dataset=train_dataset,
    batch_size=batch_size,
    shuffle=True,
    num_workers=num_workers,
    drop_last=True
)

val_dataloader=DataLoader(
    dataset=val_dataset,
    batch_size=batch_size,
    shuffle=False,
    num_workers=num_workers,
    drop_last=False
)

test_dataloader=DataLoader(
    dataset=test_dataset,
    batch_size=batch_size,
    shuffle=False,
    num_workers=num_workers,
    drop_last=False
)

In [ ]:
print("Train Loader:")
for input_batch, target_batch in train_dataloader:
    pass

print(f'Input batch dimension: {input_batch.shape}')
print(f'Target batch dimension: {target_batch.shape}')

In [ ]:
print(f'{len(train_dataloader)} training batches')
print(f'{len(val_dataloader)} validation batches')
print(f'{len(test_dataloader)} test batches')

#### Model Initialization with pretrained weights

In [ ]:
CHOOSE_MODEL="gpt2-small (124M)"
INPUT_PROMPT="Every effort moves"

BASE_CONFIG={
    "vocab_size": 50257,
    "context_length": 1024,
    "drop_rate": 0.0,
    "qkv_bias": True
}

model_configs={
  "gpt2-small (124M)": {"emb_dim": 768, "n_layers": 12, "n_heads": 12},
  "gpt2-medium (355M)": {"emb_dim": 1024, "n_layers": 24, "n_heads": 16},
  "gpt2-large (774M)": {"emb_dim": 1280, "n_layers": 36, "n_heads": 20},
  "gpt2-xl (1558M)": {"emb_dim": 1600, "n_layers": 48, "n_heads": 25}
}

BASE_CONFIG.update(model_configs[CHOOSE_MODEL])

assert train_dataset.max_length<=BASE_CONFIG["context_length"], (
    f'Dataset length {train_dataset.max_length} exceeds model context length {BASE_CONFIG["context_length"]}',
    f'Please set max_length to {BASE_CONFIG["context_length"]} or lower.'
)

In [ ]:
import torch
import torch.nn as nn

class GPTModel(nn.Module):
  def __init__(self, cfg):
    super().__init__()
    self.tok_emb=nn.Embedding(cfg["vocab_size"], cfg["emb_dim"])
    self.pos_emb=nn.Embedding(cfg["context_length"], cfg["emb_dim"])
    self.drop_emb=nn.Dropout(cfg["drop_rate"])

    self.trf_blocks=nn.Sequential(
      *[TransformerBlock(cfg) for _ in range(cfg["n_layers"])]
    )

    self.final_norm=LayerNorm(cfg["emb_dim"])
    self.out_head=nn.Linear(
      cfg["emb_dim"], cfg["vocab_size"], bias=False
    )

  def forward(self, in_idx):
    batch_size, seq_len=in_idx.shape
    tok_embeds=self.tok_emb(in_idx)
    pos_embeds=self.pos_emb(torch.arange(seq_len, device=in_idx.device))
    x=tok_embeds+pos_embeds
    x=self.drop_emb(x)
    x=self.trf_blocks(x)
    x=self.final_norm(x)
    logits=self.out_head(x)
    return logits
  
class TransformerBlock(nn.Module):
  def __init__(self, cfg):
    super().__init__()
    self.att=MultiHeadAttention(
      d_in=cfg["emb_dim"],
      d_out=cfg["emb_dim"],
      context_length=cfg["context_length"],
      dropout=cfg["drop_rate"],
      num_heads=cfg["n_heads"],
      qkv_bias=cfg["qkv_bias"]
    )
    self.ff=FeedForward(cfg)
    self.norm1=LayerNorm(cfg["emb_dim"])
    self.norm2=LayerNorm(cfg["emb_dim"])
    self.drop_shortcut=nn.Dropout(cfg["drop_rate"])

  def forward(self, x):
    # Shortcut connection for attention block
    shortcut=x
    # Every row in the input has 0 mean and 1 variance
    x=self.norm1(x)
    # We get the context vector of [batch_size, num_tokens, emb_dim]
    x=self.att(x)
    # Dropout layer to improve efficiency
    x=self.drop_shortcut(x)
    # Creating shortcut connection
    x=x+shortcut

    # Shortcut for feed forward block
    shortcut=x
    x=self.norm2(x)
    x=self.ff(x)
    x=self.drop_shortcut(x)
    x=x+shortcut

    return x
  
import torch.nn as nn

class MultiHeadAttention(nn.Module):
  def __init__(self, d_in, d_out, context_length, dropout, num_heads, qkv_bias=False):
    super().__init__()
    assert (d_out%num_heads==0), \
    "d_out must be divisible by num_heads"

    self.d_out=d_out
    self.num_heads=num_heads
    self.head_dim=d_out//num_heads

    self.w_query=nn.Linear(d_in, d_out, bias=qkv_bias)
    self.w_key=nn.Linear(d_in, d_out, bias=qkv_bias)
    self.w_value=nn.Linear(d_in, d_out, bias=qkv_bias)
    # Linear layer to combine head outputs
    self.out_proj=nn.Linear(d_out, d_out)
    self.dropout=nn.Dropout(dropout)
    self.register_buffer('mask', torch.triu(torch.ones(context_length, context_length), diagonal=1))

  def forward(self, x):
    b, num_tokens, d_in=x.shape

    queries=self.w_query(x)
    keys=self.w_key(x)
    values=self.w_value(x)

    # We implicitly split the matrix by adding a `num_heads` dimension
    # Unroll last dimension: (b, num_tokens, d_out) -> (b, num_tokens, num_heads, head_dim)
    keys=keys.view(b, num_tokens, self.num_heads, self.head_dim)
    queries=queries.view(b, num_tokens, self.num_heads, self.head_dim)
    values=values.view(b, num_tokens, self.num_heads, self.head_dim)

    # Transpose: (b, num_tokens, num_heads, head_dim) -> (b, num_heads, num_tokens, head_dim)
    keys=keys.transpose(1, 2)
    queries=queries.transpose(1, 2)
    values=values.transpose(1, 2)

    # Computing attention scores
    attn_scores=torch.matmul(queries, keys.transpose(2, 3))

    # Original mask truncated to the number of tokens and converted to bool
    masked_bool=self.mask.bool()[:num_tokens, :num_tokens]

    # Using the mask to fill the attention scores
    masked_attn_scores=attn_scores.masked_fill_(masked_bool, -torch.inf)

    # Calculating the attention weights
    attn_weights=torch.softmax(masked_attn_scores/keys.shape[-1]**0.5, dim=-1)

    # Feeding attention weights to the dropout layer
    attn_weights=self.dropout(attn_weights)

    # Calculating context vectors
    context_vecs=(attn_weights@values).transpose(1, 2) # To get the original dimensions

    # Combining heads where self.d_out=num_heads*head_dim
    # contiguous - to make sure the reshaped matrices are in same blocks of memory
    context_vecs=context_vecs.contiguous().view(b, num_tokens, self.d_out)
    context_vecs=self.out_proj(context_vecs)

    return context_vecs
  
class LayerNorm(nn.Module):
  def __init__(self, emb_dim):
    super().__init__()
    self.eps=1e-5
    self.scale=nn.Parameter(torch.ones(emb_dim))
    self.shift=nn.Parameter(torch.zeros(emb_dim))

  def forward(self, x):
    mean=x.mean(dim=-1, keepdim=True)
    var=x.var(dim=-1, keepdim=True, unbiased=False)
    # To prevent from zero division
    norm_x=(x-mean)/torch.sqrt(var+self.eps)
    return self.scale*norm_x+self.shift
  
class GELU(nn.Module):
  def __init__(self):
    super().__init__()
  
  def forward(self, x):
    gelu=0.5*x*(1+torch.tanh(torch.sqrt(torch.tensor(2.0/torch.pi, device=x.device))*(x+(0.044715*torch.pow(x, 3)))))
    return gelu
  
class FeedForward(nn.Module):
  def __init__(self, cfg):
    super().__init__()
    self.layers=nn.Sequential(
      # Expansion
      nn.Linear(cfg["emb_dim"], 4*cfg["emb_dim"]),
      # Activation
      GELU(),
      # Contraction
      nn.Linear(4*cfg["emb_dim"], cfg["emb_dim"])
    )

  def forward(self, x):
    return self.layers(x)
  

import numpy as np

def load_weights_into_gpt(gpt, params):
  gpt.pos_emb.weight=assign(gpt.pos_emb.weight, params["wpe"])
  gpt.tok_emb.weight=assign(gpt.tok_emb.weight, params["wte"])

  for b in range(len(params["blocks"])):
    q_w, k_w, v_w=np.split(
      params["blocks"][b]["attn"]["c_attn"]["w"], 3, axis=-1)
    gpt.trf_blocks[b].att.w_query.weight=assign(
      gpt.trf_blocks[b].att.w_query.weight, q_w.T
    )
    gpt.trf_blocks[b].att.w_key.weight=assign(
      gpt.trf_blocks[b].att.w_key.weight, k_w.T
    )
    gpt.trf_blocks[b].att.w_value.weight=assign(
      gpt.trf_blocks[b].att.w_value.weight, v_w.T
    )

    q_b, k_b, v_b=np.split(
      params["blocks"][b]["attn"]["c_attn"]["b"], 3, axis=-1)
    gpt.trf_blocks[b].att.w_query.bias=assign(
      gpt.trf_blocks[b].att.w_query.bias, q_b
    )
    gpt.trf_blocks[b].att.w_key.bias=assign(
      gpt.trf_blocks[b].att.w_key.bias, k_b
    )
    gpt.trf_blocks[b].att.w_value.bias=assign(
      gpt.trf_blocks[b].att.w_value.bias, v_b
    )

    gpt.trf_blocks[b].att.out_proj.weight=assign(
      gpt.trf_blocks[b].att.out_proj.weight,
      params["blocks"][b]["attn"]["c_proj"]["w"].T
    )
    gpt.trf_blocks[b].att.out_proj.bias=assign(
      gpt.trf_blocks[b].att.out_proj.bias,
      params["blocks"][b]["attn"]["c_proj"]["b"]
    )

    gpt.trf_blocks[b].ff.layers[0].weight=assign(
      gpt.trf_blocks[b].ff.layers[0].weight,
      params["blocks"][b]["mlp"]["c_fc"]["w"].T
    )
    gpt.trf_blocks[b].ff.layers[0].bias=assign(
      gpt.trf_blocks[b].ff.layers[0].bias,
      params["blocks"][b]["mlp"]["c_fc"]["b"]
    )
    gpt.trf_blocks[b].ff.layers[2].weight=assign(
      gpt.trf_blocks[b].ff.layers[2].weight,
      params["blocks"][b]["mlp"]["c_proj"]["w"].T
    )
    gpt.trf_blocks[b].ff.layers[2].bias=assign(
      gpt.trf_blocks[b].ff.layers[2].bias,
      params["blocks"][b]["mlp"]["c_proj"]["b"]
    )

    gpt.trf_blocks[b].norm1.scale=assign(
      gpt.trf_blocks[b].norm1.scale,
      params["blocks"][b]["ln_1"]["g"]
    )
    gpt.trf_blocks[b].norm1.shift=assign(
      gpt.trf_blocks[b].norm1.shift,
      params["blocks"][b]["ln_1"]["b"]
    )
    gpt.trf_blocks[b].norm2.scale=assign(
      gpt.trf_blocks[b].norm2.scale,
      params["blocks"][b]["ln_2"]["g"]
    )
    gpt.trf_blocks[b].norm2.shift=assign(
      gpt.trf_blocks[b].norm2.shift,
      params["blocks"][b]["ln_2"]["b"]
    )

  gpt.final_norm.scale=assign(gpt.final_norm.scale, params["g"])
  gpt.final_norm.shift=assign(gpt.final_norm.shift, params["b"])
  gpt.out_head.weight=assign(gpt.out_head.weight, params["wte"])

def assign(left, right):
  if left.shape!=right.shape:
    raise ValueError(f'Shape mismatch. Left Shape: {left.shape}, Right Shape: {right.shape}')
  return torch.nn.Parameter(torch.tensor(right, dtype=left.dtype, device=left.device))


def generate_text_simple(model, idx, max_new_tokens, context_size):
  # idx is (batch_size, num_tokens) array of indices in current context
  for _ in range(max_new_tokens):
    # Crop current context if it exceeds the supported context size
    '''For example, 
    Case 1: If LLM supports only 5 tokens and context size is
    10, then only the last 5 tokens are used as context.
    Case 2: If LLM supports 8 tokens and context size is 5, then
    only the last 5 tokens are used as context.'''
    idx_cond=idx[:, -context_size:]

    # Get output tensors - (batch_size, num_tokens, vocab_size)
    with torch.no_grad():
      logits=model(idx_cond)

    # Extract last vector
    logits=logits[:, -1, :]

    # Apply softmax to get probabilities - (batch_size, vocab_size)
    probs=torch.softmax(logits, dim=-1)

    # Get the idx of the vocab entry with the highest probability value
    idx_next=torch.argmax(probs, dim=-1, keepdim=True)
    # (batch_size, 1)

    # Append sampled index to the running sequence
    idx=torch.cat((idx, idx_next), dim=1)
    # (batch_size, num_tokens+1)

  return idx

def text_to_token_ids(text, tokenizer):
  encoded_text=tokenizer.encode(text, allowed_special={'<|endoftext|>'})
  encoded_tensor=torch.tensor(encoded_text).unsqueeze(0)
  return encoded_tensor

def token_ids_to_text(token_ids, tokenizer):
  flat=token_ids.squeeze(0)
  decoded_text=tokenizer.decode(flat.tolist())
  return decoded_text

In [ ]:
model_size=CHOOSE_MODEL.split(" ")[-1].lstrip("(").rstrip(")")

from gpt_download import download_and_load_gpt2

settings, params=download_and_load_gpt2(
    model_size=model_size,
    models_dir="gpt2"
)

model=GPTModel(BASE_CONFIG)
load_weights_into_gpt(model, params)
model.eval()

In [ ]:
token_ids=generate_text_simple(
    model=model,
    idx=text_to_token_ids(INPUT_PROMPT, tokenizer),
    max_new_tokens=40,
    context_size=BASE_CONFIG["context_length"]
)

print(token_ids_to_text(token_ids, tokenizer))

- There is no need to finetune all the layers of the GPT architecture since the initial layers which are used to extract features from the input text are already trained on a large corpus of text.

- The only layers which we finetune are:
  * Final output head
  * Final transformer block
  * Final layer norm block

- We replace the original output layer, which maps the hidden representation to a vocabulary of 50,257, with a small output layer that maps to 2 classes: 0(not spam) and 1(spam).

- A more general approach is to take number of output heads which matches the number of classes.

In [ ]:
print(model)

In [ ]:
for param in model.parameters():
    param.requires_grad=False

In [ ]:
torch.manual_seed(123)

num_classes=2
# Adding a classification head
model.out_head=nn.Linear(in_features=BASE_CONFIG["emb_dim"], out_features=num_classes)

In [ ]:
for param in model.trf_blocks[-1].parameters():
    param.requires_grad=True

for param in model.final_norm.parameters():
    param.requires_grad=True

In [ ]:
inputs=tokenizer.encode("Do you have time")
inputs=torch.tensor(inputs).unsqueeze(0)

print(f'Inputs: {inputs}')
print(f'Inputs dimension: {inputs.shape}')

In [ ]:
with torch.no_grad():
    logits=model(inputs)

print(f'Outputs:\n{logits}')
print(f'Outputs dimension: {logits.shape}')

In [ ]:
print(f'Last output token: {logits[:, -1, :]}')

#### Calculating Classification Loss and Accuracy

In [ ]:
torch.set_printoptions(sci_mode=False)

probs=torch.softmax(logits[:, -1, :], dim=-1)
print(probs)
label=torch.argmax(probs)
print(f'Class Label: {label}')

In [ ]:
def calc_accuracy_loader(data_loader, model, device, num_batches=None):
    model.eval()
    correct_pred, num_examples=0, 0

    if num_batches is None:
        num_batches=len(data_loader)
    else:
        num_batches=min(num_batches, len(data_loader))

    for i, (input_batch, target_batch) in enumerate(data_loader):
        if i<num_batches:
            input_batch=input_batch.to(device)
            target_batch=target_batch.to(device)

            with torch.no_grad():
                logits=model(input_batch)[:, -1, :]

            predicted_labels=torch.argmax(logits, dim=-1)
            num_examples+=predicted_labels.shape[0]
            correct_pred+=(predicted_labels==target_batch).sum().item()
        else:
            break

    return correct_pred/num_examples

In [ ]:
device=torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

torch.manual_seed(123)

training_accuracy=calc_accuracy_loader(train_dataloader, model, device, num_batches=10)
validation_accuracy=calc_accuracy_loader(val_dataloader, model, device, num_batches=10)
test_accuracy=calc_accuracy_loader(test_dataloader, model, device, num_batches=10)

print(f'Training Accuracy: {training_accuracy*100:.2f}%')
print(f'Validation Accuracy: {validation_accuracy*100:.2f}%')
print(f'Test Accuracy: {test_accuracy*100:.2f}%')

In [ ]:
def calc_loss_batch(input_batch, target_batch, model, device):
    input_batch, target_batch=input_batch.to(device), target_batch.to(device)
    logits=model(input_batch)[:, -1, :]
    loss=nn.functional.cross_entropy(logits, target_batch)
    return loss

In [ ]:
def calc_loss_loader(data_loader, model, device, num_batches=None):
    model.eval()
    total_loss=0.0

    if num_batches is None:
        num_batches=len(data_loader)
    else:
        num_batches=min(num_batches, len(data_loader))

    for i, (input_batch, target_batch) in enumerate(data_loader):
        if i<num_batches:
            batch_loss=calc_loss_batch(input_batch, target_batch, model, device)
            total_loss+=batch_loss.item()
        else:
            break
        
    return total_loss/num_batches

In [ ]:
with torch.no_grad():
    train_loss=calc_loss_loader(train_dataloader, model, device, num_batches=5)
    validation_loss=calc_loss_loader(val_dataloader, model, device, num_batches=5)
    test_loss=calc_loss_loader(test_dataloader, model, device, num_batches=5)

print(f'Training Loss: {train_loss:.4f}')
print(f'Validation Loss: {validation_loss:.4f}')
print(f'Test Loss: {test_loss:.4f}')

In [ ]:
def train_classifier_simple(model, train_loader, val_loader, optimizer,
                            device, num_epochs, eval_freq, eval_iter):
    train_losses, val_losses=[], []
    train_accs, val_accs=[], []
    examples_seen, global_step=0, -1

    for epoch in range(num_epochs):
        model.train()
        for input_batch, target_batch in train_loader:
            optimizer.zero_grad()
            loss=calc_loss_batch(input_batch, target_batch, model, device)
            loss.backward()
            optimizer.step()
            examples_seen+=input_batch.shape[0]
            global_step+=1

            if global_step%eval_freq==0:
                train_loss, val_loss = evaluate_model(model, train_loader, val_loader, device, eval_iter)
                train_losses.append(train_loss)
                val_losses.append(val_loss)
                print(f'Epoch {epoch+1}: Step {global_step:06d}: Train Loss: {train_loss:.4f}, Val Loss: {val_loss:.4f}')

        train_acc=calc_accuracy_loader(train_loader, model, device, eval_iter)
        val_acc=calc_accuracy_loader(val_loader, model, device, eval_iter)
        print(f'Training accuracy: {train_acc*100:.2f}%, Validation accuracy: {val_acc*100:.2f}%')
        train_accs.append(train_acc)
        val_accs.append(val_acc)

    return train_losses, val_losses, train_accs, val_accs, examples_seen

def evaluate_model(model, train_loader, val_loader, device, eval_iter):
  model.eval()
  with torch.no_grad():
    train_loss=calc_loss_loader(train_loader, model, device, num_batches=eval_iter)
    val_loss=calc_loss_loader(val_loader, model, device, num_batches=eval_iter)
  model.train()
  return train_loss, val_loss

In [ ]:
import time
torch.manual_seed(123)

start_time=time.time()
optimizer=torch.optim.AdamW(model.parameters(), lr=5e-5, weight_decay=0.1)
num_epochs=5

train_losses, val_losses, train_accs, val_accs, examples_seen=train_classifier_simple(
    model=model,
    train_loader=train_dataloader,
    val_loader=val_dataloader,
    optimizer=optimizer,
    device=device,
    num_epochs=num_epochs,
    eval_freq=50,
    eval_iter=5
)

end_time=time.time()
execution_time=(end_time-start_time)/60
print(f'Training completed in {execution_time:.2f} minutes.')

In [ ]:
import matplotlib.pyplot as plt

def plot_values(epochs_seen, examples_seen, train_values, val_values, label="loss"):
    fig, ax1=plt.subplots(figsize=(5, 3))

    ax1.plot(epochs_seen, train_values, label=f'Training {label}', color='blue', marker='o')
    ax1.plot(epochs_seen, val_values, label=f'Validation {label}', color='orange', marker='o')
    ax1.set_xlabel("Epochs")
    ax1.set_ylabel(label.capitalize())
    ax1.legend()

    ax2=ax1.twiny()
    ax2.plot(examples_seen, train_values, alpha=0)
    ax2.set_xlabel("Examples seen")

    fig.tight_layout()
    plt.show()

In [ ]:
epochs_tensor=torch.linspace(0, num_epochs, len(train_losses))
examples_seen_tensor=torch.linspace(0, examples_seen, len(train_losses))

plot_values(epochs_tensor, examples_seen_tensor, train_losses, val_losses)

In [ ]:
epochs_tensor=torch.linspace(0, num_epochs, len(train_accs))
examples_seen_tensor=torch.linspace(0, examples_seen, len(train_accs))

plot_values(epochs_tensor, examples_seen_tensor, train_accs, val_accs, label="accuracy")

In [ ]:
training_accuracy=calc_accuracy_loader(train_dataloader, model, device, num_batches=10)
validation_accuracy=calc_accuracy_loader(val_dataloader, model, device, num_batches=10)
test_accuracy=calc_accuracy_loader(test_dataloader, model, device, num_batches=10)

print(f'Training Accuracy: {training_accuracy*100:.2f}%')
print(f'Validation Accuracy: {validation_accuracy*100:.2f}%')
print(f'Test Accuracy: {test_accuracy*100:.2f}%')

#### Using LLM as a Spam Classifier

In [ ]:
def classify_review(text, model, tokenizer, device, max_length=None, pad_token_id=50256):
    model.eval()

    input_ids=tokenizer.encode(text)
    supported_context_length=model.pos_emb.weight.shape[0]

    input_ids=input_ids[:min(max_length, supported_context_length)]
    input_ids+=[pad_token_id]*(max_length-len(input_ids))
    input_tensor=torch.tensor(input_ids, device=device).unsqueeze(0)

    with torch.no_grad():
        logits=model(input_tensor)[:, -1, :]
    predicted_label=torch.argmax(logits, dim=-1).item()

    return "spam" if predicted_label==1 else "ham"

In [ ]:
text=(
    "You are a winner you have specially"
    "selected to receive $1000 cash or a $2000 award."
)

print(classify_review(
    text=text,
    model=model,
    tokenizer=tokenizer,
    device=device,
    max_length=train_dataset.max_length
))

In [ ]:
torch.save(model.state_dict(), "spam_classifier.pth")

In [ ]:
model_state_dict=torch.load("spam_classifier.pth")
model.load_state_dict(model_state_dict)